# Capítulo 7b: Agentes de IA con Ollama (modelos locales)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caracena/apunte-analitica-textual/blob/main/capitulos/clase7-agentes-ia-ollama.ipynb)

## Objetivos de aprendizaje

- Conectar Python con un modelo de lenguaje servido **localmente** mediante [Ollama](https://ollama.com).
- Construir un agente de IA **real** (no simulado) sobre el patrón ReAct.
- Definir herramientas y dejar que el LLM decida cuándo y cómo usarlas.
- Implementar memoria conversacional y un pipeline RAG con embeddings locales.

> **Requisito previo:** Este notebook asume que tienes **Ollama corriendo** en tu computador
> (`ollama serve`) y los modelos descargados. El modelo de razonamiento se auto-detecta;
> el de embeddings (`nomic-embed-text`) es **necesario** para la sección de RAG.
>
> ```bash
> # Instalar Ollama: https://ollama.com/download
> ollama pull qwen3.5:2b          # modelo de razonamiento (o el que prefieras)
> ollama pull embeddinggemma  # embeddings para el RAG
> ```

## 7b.1 ¿Por qué un modelo local?

A diferencia de las APIs en la nube (OpenAI, Anthropic, etc.), **Ollama** ejecuta los modelos
en tu propia máquina. Esto tiene ventajas concretas para experimentar y para casos sensibles:

| Aspecto | API en la nube | Ollama (local) |
|---------|----------------|----------------|
| **Costo** | Por token | Gratis (usa tu hardware) |
| **Privacidad** | Datos salen del equipo | Datos nunca salen del equipo |
| **Conexión** | Requiere internet | Funciona sin internet |
| **Latencia** | Red + servidor | Depende de tu GPU/CPU |
| **Modelos** | Los del proveedor | Cualquiera del catálogo de Ollama |

Ollama expone una API HTTP local en `http://localhost:11434`. Podemos hablar con ella
mediante la librería oficial `ollama` o con peticiones HTTP directas.

In [ ]:
# Instalar el cliente de Ollama (la app/servidor se instala aparte)
# %pip install ollama

import ollama

# Verificar que el servidor responde y listar modelos disponibles
try:
    modelos = ollama.list()
    print("Ollama está corriendo. Modelos disponibles:")
    for m in modelos["models"]:
        nombre = m.get("model", m.get("name", "?"))
        tam_gb = m.get("size", 0) / 1e9
        print(f"  - {nombre} ({tam_gb:.1f} GB)")
except Exception as e:
    print("No se pudo conectar a Ollama. ¿Está corriendo 'ollama serve'?")
    print(f"Error: {e}")

In [ ]:
# Elegimos el modelo a usar. Puedes fijarlo a mano o dejar que se auto-detecte.
MODELO_LLM = "qwen3.5:2b"          # p.ej. "llama3.2"; None = auto-detectar
MODELO_EMBED = "embeddinggemma"  # modelo de embeddings (si lo descargaste)

print(f"Usando modelo de razonamiento: {MODELO_LLM}")

# Primera llamada: un chat simple para confirmar que todo funciona
respuesta = ollama.chat(
    model=MODELO_LLM,
    messages=[
        {"role": "user", "content": "En una frase, ¿qué es un agente de IA?"}
    ],
)
print(respuesta["message"]["content"])

## 7b.2 Definir herramientas para el agente

Una herramienta es simplemente una **función de Python** que el agente puede invocar.
Para que el LLM las use bien, además de implementarlas necesitamos **describirlas**
(nombre, propósito y parámetros) en el formato de *tool calling* que entiende Ollama.

In [ ]:
import json
from datetime import datetime


def herramienta_calculadora(expresion: str) -> str:
    """Evalúa una expresión matemática y retorna el resultado."""
    try:
        # Entorno restringido: solo operaciones aritméticas
        resultado = eval(expresion, {"__builtins__": {}}, {})
        return f"{resultado}"
    except Exception as e:
        return f"Error al evaluar '{expresion}': {e}"


def herramienta_fecha() -> str:
    """Retorna la fecha y hora actual del sistema."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def herramienta_buscar_producto(consulta: str) -> str:
    """Busca un producto en el catálogo y devuelve precio, stock y marca."""
    productos = {
        "laptop": {"precio": 899990, "stock": 15, "marca": "TechPro"},
        "teclado": {"precio": 45990, "stock": 50, "marca": "KeyMax"},
        "mouse": {"precio": 25990, "stock": 80, "marca": "ClickPro"},
        "monitor": {"precio": 349990, "stock": 8, "marca": "ViewMax"},
    }
    consulta_lower = consulta.lower()
    for nombre, info in productos.items():
        if nombre in consulta_lower:
            return json.dumps({nombre: info}, ensure_ascii=False)
    return "Producto no encontrado en el catálogo."


# Registro: nombre -> función ejecutable
HERRAMIENTAS = {
    "calculadora": herramienta_calculadora,
    "fecha": herramienta_fecha,
    "buscar_producto": herramienta_buscar_producto,
}

print("Herramientas registradas:")
for nombre, func in HERRAMIENTAS.items():
    print(f"  - {nombre}: {func.__doc__}")

In [ ]:
# Esquema de las herramientas en el formato de "tools" de Ollama (estilo OpenAI).
# El LLM lee estas descripciones para decidir cuál llamar y con qué argumentos.
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "calculadora",
            "description": "Evalúa una expresión matemática aritmética, por ejemplo '899990 * 3 * 1.19'.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expresion": {
                        "type": "string",
                        "description": "La expresión matemática a evaluar.",
                    }
                },
                "required": ["expresion"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "fecha",
            "description": "Devuelve la fecha y hora actual del sistema. No requiere argumentos.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "buscar_producto",
            "description": "Busca un producto (laptop, teclado, mouse, monitor) y devuelve su precio, stock y marca.",
            "parameters": {
                "type": "object",
                "properties": {
                    "consulta": {
                        "type": "string",
                        "description": "Nombre del producto a buscar.",
                    }
                },
                "required": ["consulta"],
            },
        },
    },
]

## 7b.3 Un agente ReAct real con Ollama

Ahora construimos el ciclo del agente. A diferencia del capítulo anterior (donde
los pasos estaban escritos a mano), aquí **el modelo decide**:

1. Recibe el objetivo y la lista de herramientas.
2. Si necesita información, responde con una **llamada a herramienta** (`tool_calls`).
3. Ejecutamos la herramienta y le devolvemos el resultado como un mensaje `role="tool"`.
4. El modelo repite hasta que puede responder sin más herramientas.

Este es el patrón **ReAct** (Reasoning + Acting) implementado sobre *tool calling* nativo.

In [ ]:
class AgenteOllama:
    """Agente ReAct que usa un modelo local de Ollama con tool calling."""

    def __init__(self, modelo, herramientas, tools_schema, max_iter=6, verbose=True):
        self.modelo = modelo
        self.herramientas = herramientas
        self.tools_schema = tools_schema
        self.max_iter = max_iter
        self.verbose = verbose

    def ejecutar(self, objetivo, sistema=None):
        mensajes = []
        if sistema:
            mensajes.append({"role": "system", "content": sistema})
        mensajes.append({"role": "user", "content": objetivo})

        for iteracion in range(1, self.max_iter + 1):
            respuesta = ollama.chat(
                model=self.modelo,
                messages=mensajes,
                tools=self.tools_schema,
            )
            msg = respuesta["message"]
            mensajes.append(msg)

            tool_calls = msg.get("tool_calls")
            if not tool_calls:
                # El modelo respondió sin pedir herramientas: terminamos.
                if self.verbose:
                    print(f"[Iteración {iteracion}] Respuesta final.\n")
                return msg["content"], mensajes

            # Ejecutar cada herramienta solicitada y devolver la observación.
            for tc in tool_calls:
                nombre = tc["function"]["name"]
                args = tc["function"].get("arguments", {}) or {}
                if isinstance(args, str):
                    try:
                        args = json.loads(args)
                    except json.JSONDecodeError:
                        args = {}

                if self.verbose:
                    print(f"[Iteración {iteracion}] Acción: {nombre}({args})")

                func = self.herramientas.get(nombre)
                if func is None:
                    observacion = f"Herramienta '{nombre}' no existe."
                else:
                    try:
                        observacion = func(**args)
                    except Exception as e:
                        observacion = f"Error ejecutando {nombre}: {e}"

                if self.verbose:
                    print(f"             Observación: {observacion}")

                mensajes.append({
                    "role": "tool",
                    "name": nombre,
                    "content": str(observacion),
                })

        return "No se alcanzó una respuesta dentro del límite de iteraciones.", mensajes

In [ ]:
SISTEMA = (
    "Eres un asistente de cotizaciones. SIEMPRE usa las herramientas para obtener datos "
    "reales: usa 'buscar_producto' para precios del catálogo y 'calculadora' para cualquier "
    "operación aritmética. No inventes precios ni hagas cálculos mentalmente. "
    "Cuando tengas el resultado, responde breve y claro en español."
)

agente = AgenteOllama(MODELO_LLM, HERRAMIENTAS, TOOLS_SCHEMA, verbose=True)

objetivo = (
    "Busca el precio de la laptop en el catálogo y calcula el precio total de 3 unidades "
    "aplicando IVA del 19%."
)
print(f"Objetivo: {objetivo}\n")

respuesta_final, traza = agente.ejecutar(objetivo, sistema=SISTEMA)
print("Respuesta final del agente:")
print(respuesta_final)

### Inspeccionar la traza completa

La lista `traza` contiene todos los mensajes intercambiados: el objetivo del usuario,
las decisiones del modelo, las llamadas a herramientas y sus observaciones. Es la
"memoria de trabajo" de esta ejecución.

In [ ]:
for i, m in enumerate(traza):
    rol = m["role"] if isinstance(m, dict) else m.get("role", "?")
    if rol == "assistant" and m.get("tool_calls"):
        for tc in m["tool_calls"]:
            print(f"{i}. [assistant -> herramienta] {tc['function']['name']}({tc['function'].get('arguments')})")
    elif rol == "tool":
        print(f"{i}. [observación de {m.get('name')}] {m.get('content')}")
    elif rol == "assistant":
        print(f"{i}. [assistant] {m.get('content')}")
    else:
        contenido = m["content"] if isinstance(m, dict) else ""
        print(f"{i}. [{rol}] {contenido}")

## 7b.4 Memoria conversacional

Para que el agente recuerde el contexto entre turnos, basta con **acumular los mensajes**.
Aquí encapsulamos un chat con memoria de corto plazo sobre el mismo modelo local.

In [ ]:
class ChatConMemoria:
    """Conversación multi-turno que mantiene el historial completo."""

    def __init__(self, modelo, sistema=None):
        self.modelo = modelo
        self.mensajes = []
        if sistema:
            self.mensajes.append({"role": "system", "content": sistema})

    def enviar(self, texto):
        self.mensajes.append({"role": "user", "content": texto})
        respuesta = ollama.chat(model=self.modelo, messages=self.mensajes)
        contenido = respuesta["message"]["content"]
        self.mensajes.append({"role": "assistant", "content": contenido})
        return contenido


chat = ChatConMemoria(
    MODELO_LLM,
    sistema="Eres un tutor de analítica textual. Responde breve y en español.",
)

print("Usuario: Mi nombre es Ana y estudio NLP.")
print("Bot:", chat.enviar("Mi nombre es Ana y estudio NLP."))
print()
print("Usuario: ¿Cómo me llamo y qué estudio?")
print("Bot:", chat.enviar("¿Cómo me llamo y qué estudio?"))

## 7b.5 RAG con embeddings locales

Ollama también puede generar **embeddings**.
Construimos un pipeline RAG donde:

1. Indexamos los documentos como vectores con el modelo de embeddings.
2. Ante una pregunta, recuperamos los pasajes más similares (similitud coseno con `cosine_similarity` de scikit-learn).
3. Inyectamos ese contexto en el prompt del LLM para una respuesta **fundamentada**.

> Este pipeline **requiere** un modelo de embeddings descargado en Ollama
> (`ollama pull embeddingmodel`). Si no está disponible, la indexación fallará.

In [ ]:
# %pip install numpy
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

class RAGLocal:
    """Pipeline RAG local que indexa con embeddings de Ollama."""

    def __init__(self, base_conocimiento, modelo_llm=MODELO_LLM, modelo_embed=MODELO_EMBED):
        self.modelo_llm = modelo_llm
        self.modelo_embed = modelo_embed
        self.documentos = list(base_conocimiento)

        print(f"Indexando con embeddings de Ollama ({modelo_embed})...")
        self.vectores = np.vstack([self._embed_ollama(d) for d in self.documentos])
        print(f"Listo: {len(self.documentos)} documentos indexados.")

    def _embed_ollama(self, texto):
        r = ollama.embeddings(model=self.modelo_embed, prompt=texto)
        return np.array(r["embedding"], dtype=np.float32)

    def recuperar(self, pregunta, top_k=2):
        q = self._embed_ollama(pregunta).reshape(1, -1)
        sims = cosine_similarity(q, self.vectores)[0]
        orden = np.argsort(sims)[::-1][:top_k]
        return [(self.documentos[i], float(sims[i])) for i in orden]

    def responder(self, pregunta, top_k=2):
        contextos = self.recuperar(pregunta, top_k=top_k)
        contexto_str = "\n".join(f"- {doc}" for doc, _ in contextos)
        prompt = (
            f"Usa SOLO la siguiente información para responder:\n\n{contexto_str}\n\n"
            f"Pregunta: {pregunta}\n"
            f"Si la información no alcanza, dilo claramente."
        )
        respuesta = ollama.chat(
            model=self.modelo_llm,
            messages=[{"role": "user", "content": prompt}],
        )
        return respuesta["message"]["content"], contextos

In [ ]:
base = [
    "La empresa TechCorp fue fundada en 2015 en Santiago de Chile.",
    "TechCorp ofrece soluciones de inteligencia artificial para empresas.",
    "El producto principal de TechCorp es un chatbot llamado AsistenteIA.",
    "AsistenteIA puede manejar hasta 10.000 consultas simultáneas.",
    "Los planes de TechCorp van desde $99.000 hasta $999.000 mensuales.",
    "TechCorp tiene oficinas en Chile, Colombia y México.",
]

rag = RAGLocal(base)

pregunta = "¿Cuánto cuestan los servicios de TechCorp?"
respuesta, contextos = rag.responder(pregunta)

print(f"Pregunta: {pregunta}\n")
print("Contextos recuperados:")
for doc, sim in contextos:
    print(f"  (sim={sim:.3f}) {doc}")
print(f"\nRespuesta del LLM:\n{respuesta}")

## Resumen

En este capítulo conectamos la teoría de agentes con una implementación **real y local**:

- **Ollama** nos permite ejecutar LLMs en nuestro propio equipo, sin costos por token ni
  enviar datos a la nube.
- Un **agente ReAct** funciona cuando el modelo decide por sí mismo qué herramienta usar,
  mediante *tool calling* nativo, y nosotros ejecutamos y devolvemos las observaciones.
- La **memoria conversacional** es simplemente el historial acumulado de mensajes.
- **RAG** con embeddings locales (`nomic-embed-text`) entrega respuestas fundamentadas en
  una base de conocimiento propia.

### Ideas para experimentar

- Cambia `MODELO_LLM` por otro modelo (`qwen2.5`, `mistral`, `phi3`) y compara el
  comportamiento del agente.
- Agrega una herramienta nueva (por ejemplo, consultar el clima o leer un archivo).
- Aumenta `top_k` en el RAG y observa cómo cambia la calidad de las respuestas.